# Nu-Classifier Model Analysis

Loads predictions from `predict_mc.py` and `predict_exp.py` and produces:
1. Score distribution histograms (muatm / nuatm / nue2 / exp)
2. Suppression vs efficiency curve
3. UMAP of encoder representations (sampled, encoded on-the-fly)
4. Matching utility: enrich MC predictions with truth info from source h5

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

MODEL_TAG = "260507_2121_da_nu_classifier_h5s0_lambda0.01_thr0.8"
#MODEL_TAG = "260507_2121_da_nu_classifier_h5s0_lambda0.01_thr0.8"

# ── Paths — edit these ────────────────────────────────────────────────
MC_PREDS_H5  = PROJECT_ROOT / f"inference/nu_classifier_model/results/mc_preds_{MODEL_TAG}.h5"
EXP_PREDS_H5 = PROJECT_ROOT / f"inference/nu_classifier_model/results/exp_preds_{MODEL_TAG}.h5"
EXP_KEY = "exp" #"exp_reco"
CHECKPOINT   = PROJECT_ROOT / f"experiments/numu/{MODEL_TAG}/best_da_model.pth"

MC_H5    = PROJECT_ROOT / "data_manager/data/h5datasets/baikal_mc_merged.h5"
#EXP_H5   = PROJECT_ROOT / "data_manager/data/h5datasets/exp_reco.h5"
EXP_H5   = PROJECT_ROOT / "data_manager/data/h5datasets/exp.h5"
PROBS_H5 = PROJECT_ROOT / "data_manager/data/h5datasets/baikal_mc_merged_probs_k_nsol_labelneq0_da_hs128_k0p0001.h5"

THRESHOLD = 0.8   # sig-noise threshold used during inference

PTYPE_COLORS = {
    "muatm_2020": "steelblue",
    "nuatm_2020": "forestgreen",
    "nue2_2020":  "darkorange",
    "exp_reco":   "crimson",
    "exp_reco_strange": "purple"
}
PTYPE_LABELS = {
    "muatm_2020": r"MC EAS",
    "nuatm_2020": r"MC $\nu_{atm}$",
    "nue2_2020":  r"MC $\nu_{cosmogenic}$",
    "exp_reco":   "EXP",
    "exp_reco_strange": "EXP big score"
}

RC = {
    "font.size":        14,
    "axes.labelsize":   16,
    "axes.titlesize":   16,
    "legend.fontsize":  13,
    "xtick.labelsize":  13,
    "ytick.labelsize":  13,
    "figure.dpi":       120,
}

## Cell 1 — Loader functions

In [ ]:
def load_baikal_mc_merged_scores(h5_path: str | Path) -> pd.DataFrame:
    """Load MC predictions from /baikal_mc_merged/{ptype}/... groups.

    Returns DataFrame with columns:
        ptype, part_key, event_id, score, n_sig_hits, n_sig_strings
    """
    chunks = []
    with h5py.File(h5_path, "r") as f:
        mc_grp = f["baikal_mc_merged"]
        for ptype in mc_grp.keys():
            if "scores" not in mc_grp[ptype]:
                continue
            for pk in tqdm(mc_grp[ptype]["scores"].keys(), desc=f'Loading {ptype} MC parts'):
                scores    = mc_grp[f"{ptype}/scores/{pk}/data"][:]
                n_sh      = mc_grp[f"{ptype}/n_sig_hits/{pk}/data"][:]
                n_ss      = mc_grp[f"{ptype}/n_sig_strings/{pk}/data"][:]
                event_ids = mc_grp[f"{ptype}/event_ids/{pk}/data"][:]
                chunks.append(pd.DataFrame({
                    "ptype":         ptype,
                    "part_key":      pk,
                    "event_id":      event_ids,
                    "score":         scores,
                    "n_sig_hits":    n_sh,
                    "n_sig_strings": n_ss,
                }))
    df = pd.concat(chunks, ignore_index=True)
    print(f"MC scores loaded: {len(df):,} events  "
          f"({df['ptype'].value_counts().to_dict()})")
    return df


def load_exp_scores(h5_path: str | Path, exp_key = EXP_KEY) -> pd.DataFrame:
    """Load exp predictions from /exp_reco/... groups.

    Returns DataFrame with columns:
        part_key, event_id, score, n_sig_hits, n_sig_strings
    """
    chunks = []
    with h5py.File(h5_path, "r") as f:
        if exp_key not in f:
            raise KeyError(f"Group '{exp_key}' not found in exp predictions h5")
        grp = f[exp_key]
        for pk in tqdm(grp["scores"].keys(), desc='Loading exp parts'):
            scores    = grp[f"scores/{pk}/data"][:]
            n_sh      = grp[f"n_sig_hits/{pk}/data"][:]
            n_ss      = grp[f"n_sig_strings/{pk}/data"][:]
            event_ids = grp[f"event_ids/{pk}/data"][:]
            chunks.append(pd.DataFrame({
                "part_key":      pk,
                "event_id":      event_ids,
                "score":         scores,
                "n_sig_hits":    n_sh,
                "n_sig_strings": n_ss,
            }))
    df = pd.concat(chunks, ignore_index=True)
    print(f"Exp scores loaded: {len(df):,} events")
    return df

In [ ]:
def add_exp_max_q(df: pd.DataFrame, exp_h5_path: str | Path) -> pd.DataFrame:
    """Add 'max_q' column (max raw amplitude per event) from exp_reco.h5.

    Uses event_ids + part_key to look up hits in the source file.
    Operates on all raw hits (before sig-noise filtering).
    """
    from collections import defaultdict
    import numpy as np

    # Group row positions by part_key
    part_to_rows = defaultdict(list)  # pk -> [(row_pos, event_id), ...]
    for row_pos, (pk, eid) in enumerate(zip(df["part_key"], df["event_id"])):
        part_to_rows[pk].append((row_pos, int(eid)))

    max_q = np.empty(len(df), dtype=np.float32)

    with h5py.File(exp_h5_path, "r") as src:
        src_key = "exp_reco" if "exp_reco" in src else "exp"
        raw = src[src_key]["raw"]
        for pk, row_events in tqdm(part_to_rows.items(), desc='Adding max Q column'):
            data      = raw[f"data/{pk}/data"][:]          # (total_hits, 5)
            ev_starts = raw[f"ev_starts/{pk}/data"][:].astype(np.int64)
            for row_pos, eid in row_events:
                s, e = int(ev_starts[eid]), int(ev_starts[eid + 1])
                max_q[row_pos] = data[s:e, 0].max() if e > s else 0.0

    df = df.copy()
    df["max_q"] = max_q
    return df

In [ ]:
with h5py.File(EXP_PREDS_H5, "r") as f:
    print(f.keys())

In [ ]:
df_exp = load_exp_scores(EXP_PREDS_H5)
df_mc  = load_baikal_mc_merged_scores(MC_PREDS_H5)
# BAD_EXP_PARTS = [
#     'part_s2020_c04_r0118', 
#     'part_s2020_c02_r0206', 
#     'part_s2020_c05_r0391', 
#     'part_s2020_c05_r0392', 
#     'part_s2020_c05_r0100'
#     ]
# df_exp = df_exp[~(df_exp['part_key'].str.contains('c01'))]

df_exp = df_exp[~df_exp['part_key'].isin(['part_s2020_c04_r0117'])]

# ## Filter events having max Q >1e4 ev
# df_exp = add_exp_max_q(df_exp, EXP_H5)
# df_exp = df_exp[df_exp['max_q'] <= 1e4]


# Quality Masks
print("Starting...")
EXP_MASK = (df_exp['n_sig_hits']>=8) & (df_exp['n_sig_strings']>=3)
print("Got EXP mask")
MC_MASK = (df_mc['n_sig_hits']>=8) & (df_mc['n_sig_strings']>=3)
print("Got MC mask...")

muatm_scores = df_mc.loc[MC_MASK & (df_mc["ptype"] == "muatm_2020"), "score"].values
nuatm_scores = df_mc.loc[MC_MASK & (df_mc["ptype"] == "nuatm_2020"), "score"].values
nue2_scores  = df_mc.loc[MC_MASK & (df_mc["ptype"] == "nue2_2020"),  "score"].values
nu_scores    = np.concatenate([nuatm_scores, nue2_scores])
exp_scores   = df_exp[EXP_MASK]["score"].values

print(f"muatm: {len(muatm_scores):,}  nuatm: {len(nuatm_scores):,}  "
     f"nue2: {len(nue2_scores):,}  exp: {len(exp_scores):,}")

## Cell 2 — Score distribution histograms

In [ ]:
plt.close()
with plt.rc_context(RC):
    fig, ax = plt.subplots(figsize=(8, 5))

    bins = np.linspace(0, 1, 51)
    kw = dict(bins=bins, density=True, histtype="step", linewidth=1.8)

    text = r'EAS'
    ax.hist(muatm_scores, label=rf"MC ${text}$  ({len(muatm_scores):,})",
            color=PTYPE_COLORS["muatm_2020"], **kw)
    text = r'\nu_{\mu}^{atmospheric}'
    ax.hist(nuatm_scores, label=fr"MC ${text}$ ({len(nuatm_scores):,})",
            color=PTYPE_COLORS["nuatm_2020"], **kw)
    text = r'\nu_{\mu}^{cosmogenic}'
    ax.hist(nue2_scores,  label=rf"MC ${text}$ ({len(nue2_scores):,})",
            color=PTYPE_COLORS["nue2_2020"],  **kw)
    ax.hist(exp_scores,   label=f"Experimental       ({len(exp_scores):,})",
            color=PTYPE_COLORS["exp_reco"],   linestyle="--", **kw)

    ax.set_yscale("log")
    ax.set_xlabel("Nu-classifier score")
    ax.set_ylabel("Density")
    ax.set_title("Nu-classifier score distributions")
    ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
cut = 0.99

print((df_exp[EXP_MASK]['score']>cut).sum() / df_exp[EXP_MASK].__len__())

df_strange = df_exp[EXP_MASK][df_exp[EXP_MASK]['score']>cut]
df_counts_strange = pd.DataFrame(df_strange['part_key'].value_counts())
df_counts = pd.DataFrame(df_exp[EXP_MASK]['part_key'].value_counts())

df_counts_strange = df_counts_strange.merge(df_counts, on='part_key', how='left', suffixes=('_cut', '_all'))
df_counts_strange['portion'] = df_counts_strange['count_cut'] / df_counts_strange['count_all']

plt.close()
df_counts_strange['portion'].apply(np.log10).hist(bins=100)
plt.show()

display(df_counts_strange.sort_values('portion', ascending=False))

## Cell 3 — Suppression vs Efficiency

In [ ]:
thresholds = np.linspace(0, 1, 500)

supp_muatm = np.array([1.0 / max((muatm_scores > t).mean(), 1e-9) for t in thresholds])
supp_exp   = np.array([1.0 / max((exp_scores   > t).mean(), 1e-9) for t in thresholds])
eff_nu     = np.array([(nu_scores > t).mean() for t in thresholds])

# Working point: 10^6 suppression of exp
TARGET_SUPP = 1e6
wp_idx = np.searchsorted(supp_muatm, TARGET_SUPP)
wp_thr = thresholds[min(wp_idx, len(thresholds) - 1)]
wp_eff = eff_nu[min(wp_idx, len(thresholds) - 1)]
wp_supp_mu = supp_muatm[min(wp_idx, len(thresholds) - 1)]
print(f"Working point: threshold={wp_thr:.3f}  "
      f"exp suppression≈{TARGET_SUPP:.0e}  "
      f"nu efficiency={wp_eff:.3f}  "
      f"muatm suppression={wp_supp_mu:.1e}")

plt.close()
with plt.rc_context(RC):
    fig, ax1 = plt.subplots(figsize=(9, 5))
    ax2 = ax1.twinx()

    ax1.semilogy(thresholds, supp_exp,   color="crimson",    lw=2,   label="EXP suppression")
    ax1.semilogy(thresholds, supp_muatm, color="steelblue",  lw=2,   label="MC μ suppression")
    ax2.plot(    thresholds, eff_nu,     color="forestgreen", lw=2, ls="--", label="ν efficiency")

    # Working point marker
    ax1.axvline(wp_thr, color="k", ls=":", lw=1.2)
    ax1.axhline(TARGET_SUPP, color="k", ls=":", lw=1.2)
    ax1.scatter([wp_thr], [TARGET_SUPP], color="k", zorder=5, s=60)
    ax1.annotate(f"  T={wp_thr:.2f}\n  eff={wp_eff:.3f}",
                 (wp_thr, TARGET_SUPP), fontsize=11)

    ax1.set_xlabel("Score threshold")
    ax1.set_ylabel("Suppression (1 / pass rate)")
    ax2.set_ylabel("ν efficiency")
    ax1.set_ylim(1, 1e8)
    ax2.set_ylim(0, 1.05)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
    fig.tight_layout()
    plt.show()

## Plot For the paper

In [ ]:
import matplotlib.ticker as ticker
from sklearn.metrics import roc_curve

# ── Precompute arrays ──
df_mc["is_neutrino"] = df_mc["ptype"].str.startswith("nu")
SCORES     = df_mc["score"].values
SIG_HITS   = df_mc["n_sig_hits"].values
SIG_S      = df_mc["n_sig_strings"].values
IS_NU      = df_mc["is_neutrino"].values
IS_MU      = ~IS_NU
ptypes_arr = df_mc["ptype"].values

pos_nue2  = (ptypes_arr == "nue2_2020")
pos_nuatm = (ptypes_arr == "nuatm_2020")

In [ ]:
# ── Config ──
REF_H          = 8
REF_S          = 3
SUPP_CLIP      = 1e8
REF_FPR_TARGET = 1e6
CUT_LABEL      = f"Signal hits≥{REF_H}, Signal strings≥{REF_S}"

def make_quality_mask(h: int, s: int) -> np.ndarray:
    if h == 0:
        return np.ones(len(df_mc), dtype=bool)
    return (SIG_HITS >= h) & (SIG_S >= s)


def balanced_nuatm_nue2_subsample(pos_nuatm, pos_nue2, quality: np.ndarray,
                                   seed: int = 42) -> np.ndarray:
    """Subsample nuatm and nue2 to equal counts after quality cut."""
    rng = np.random.default_rng(seed)
    idx_nuatm = np.where(pos_nuatm & quality)[0]
    idx_nue2  = np.where(pos_nue2  & quality)[0]
    n = min(len(idx_nuatm), len(idx_nue2))
    sel_nuatm = rng.choice(idx_nuatm, size=n, replace=False)
    sel_nue2  = rng.choice(idx_nue2,  size=n, replace=False)
    mask = np.zeros(len(ptypes_arr), dtype=bool)
    mask[sel_nuatm] = True
    mask[sel_nue2]  = True
    return mask


def get_ref_point(pos_mask: np.ndarray):
    """Return (tpr, suppression, threshold) at TPR ~ REF_FPR_TARGET."""
    quality = make_quality_mask(REF_H, REF_S)
    keep    = (pos_mask & quality) | (IS_MU & quality)
    y_true  = pos_mask[keep].astype(int)
    y_score = SCORES[keep]
    fpr_arr, tpr_arr, thr_arr = roc_curve(y_true, y_score)
    thr_arr = thr_arr[1:][::-1]
    tpr_arr = tpr_arr[1:][::-1]
    fpr_arr = fpr_arr[1:][::-1]
    inv_fpr = np.where(fpr_arr > 0, 1.0 / fpr_arr, np.nan)
    idx = min(np.searchsorted(inv_fpr, REF_FPR_TARGET), len(thr_arr) - 1)
    return float(tpr_arr[idx]), float(inv_fpr[idx]), float(thr_arr[idx])


# ── Build sample ──
quality         = make_quality_mask(REF_H, REF_S)
pos_nu_balanced = balanced_nuatm_nue2_subsample(pos_nuatm, pos_nue2, quality)
keep            = (IS_NU & pos_nu_balanced & quality) | (IS_MU & quality)

y_true  = (IS_NU & pos_nu_balanced)[keep].astype(int)
y_score = SCORES[keep]

# Event counts after cuts (for title)
n_mu_cut    = int(IS_MU[keep].sum())
n_nuatm_cut = int((pos_nuatm & pos_nu_balanced)[keep].sum())
n_nue2_cut  = int((pos_nue2  & pos_nu_balanced)[keep].sum())

N_bg  = n_mu_cut
N_sig = n_nuatm_cut + n_nue2_cut

# ── ROC ──
fpr_arr, tpr_arr, thr_arr = roc_curve(y_true, y_score)
thr_arr = thr_arr[1:][::-1]
tpr_arr = tpr_arr[1:][::-1]
fpr_arr = fpr_arr[1:][::-1]
inv_fpr = np.clip(np.where(fpr_arr > 0, 1.0 / fpr_arr, np.nan), 10, SUPP_CLIP)
clip    = (thr_arr >= 0) & (thr_arr <= 1) & (tpr_arr >= 0.5)

# ── Reference point ──
ref_tpr, ref_supp, ref_thr = get_ref_point(IS_NU & pos_nu_balanced)

# ── Plot ──
with plt.rc_context(RC):
    fig, ax = plt.subplots(figsize=(8, 6))
    c = plt.rcParams["axes.prop_cycle"].by_key()["color"][0]
    ax.plot(tpr_arr[clip], inv_fpr[clip], color=c,
            lw=3, 
            label=CUT_LABEL)

    ax.axhline(REF_FPR_TARGET, color="red", ls="--", #lw=1.2,
            label=fr"EAS Suppression = {int(REF_FPR_TARGET)}")
    ax.scatter([ref_tpr], [ref_supp], color="red", zorder=5, s=35)
    ax.annotate(
        f"EAS Suppression = {REF_FPR_TARGET:,.0f}\n"
        fr"@ $\nu$ Efficiency = {int(ref_tpr*100)}%"
        #fr"@ $\xi$ = {ref_thr:.3f}"
        ,
        xy=(ref_tpr, ref_supp),
        xytext=(ref_tpr - 0.25, ref_supp-800000),
        #fontsize=10,
        color="black", fontstyle="italic",
        arrowprops=dict(arrowstyle="->", color="red", lw=2.5), va="center",
    )

    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.yaxis.set_minor_formatter(ticker.NullFormatter())
    ax.set_xlabel(r"$\nu$ Efficiency")
    ax.set_ylabel(r"EAS Suppression")
    ax.minorticks_on()
    ax.grid(True, which="major", alpha=0.35)
    ax.grid(True, which="minor", alpha=0.18, linestyle=":")
    ax.tick_params(which="major", length=6)
    ax.tick_params(which="minor", length=3)

    fig.tight_layout()
    plt.ylim(bottom=1000)
    plt.show()

fig.savefig("SvsEff_nuclassifier.png")

In [ ]:
print(ref_thr)

## Cell 4 — UMAP of encoder representations

Samples ~10k events per domain, reloads their filtered hits from source h5,
encodes them with `model.get_feature_representation()`, then runs UMAP.

In [ ]:
from inference.nu_classifier_model.utils import load_model, predict_scores

dev = "cuda:2" if torch.cuda.is_available() else "cpu"
model, norm_config, train_config = load_model(str(CHECKPOINT), device=dev)
model.eval()
with_probs = train_config.get("model", {}).get("input_dim", 5) == 6
print(f"Model loaded on {dev}, with_probs={with_probs}")

In [ ]:
@torch.no_grad()
def _encode_features_list(
    features_list, norm_config, model, device, batch_size=256, feats_with_probs=False
) -> np.ndarray:
    """Run model.get_feature_representation() on a list of hit arrays.
    Returns (N, d_model) float32 array."""
    if feats_with_probs:
        nrm = {
            "means": norm_config["means"] + [0.75],
            "stds":  norm_config["stds"]  + [0.25],
        }
        n_feat = 6
    else:
        nrm    = norm_config
        n_feat = 5

    means = torch.tensor(nrm["means"], dtype=torch.float32, device=device)
    stds  = torch.tensor(nrm["stds"],  dtype=torch.float32, device=device)

    all_enc = []
    n = len(features_list)
    for start in tqdm(range(0, n, batch_size), desc='Encoding features'):
        batch = features_list[start:start + batch_size]
        b = len(batch)
        lengths = [min(len(f), 500) for f in batch]
        max_len = max(lengths)
        lengths_t = torch.tensor(lengths, dtype=torch.long, device=device)

        padded = torch.zeros(b, max_len, n_feat, dtype=torch.float32, device=device)
        for i, feat in enumerate(batch):
            sl = lengths[i]
            padded[i, :sl] = torch.from_numpy(feat[:sl].astype(np.float32)).to(device)

        mask = torch.arange(max_len, device=device)[None, :] < lengths_t[:, None]
        padded = torch.where(mask.unsqueeze(-1), (padded - means) / (stds + 1e-8), padded)

        enc = model.get_feature_representation({"features": padded, "lengths": lengths_t, "mask": mask})
        all_enc.append(enc.cpu().numpy())

    return np.concatenate(all_enc, axis=0)


def _load_mc_enc_feats(
    mc_preds_h5, mc_h5_path, probs_h5_path,
    ptype, threshold, model, norm_config, with_probs, device,
    n_sample=10_000, batch_size=256, seed=42,
) -> np.ndarray:
    """Sample n_sample events, load their filtered hits (prob > threshold), encode."""
    from collections import defaultdict
    rng = np.random.RandomState(seed)

    locations = []
    with h5py.File(mc_preds_h5, "r") as f:
        grp = f["baikal_mc_merged"][ptype]
        parts = list(grp["event_ids"].keys())
        parts_sel = rng.choice(len(parts), 10, replace=False)
        parts = [parts[i] for i in parts_sel.tolist()]
        for pk in parts:
            eids = grp[f"event_ids/{pk}/data"][:]
            locations.extend([(pk, int(e)) for e in eids])

    if len(locations) > n_sample:
        idx = rng.choice(len(locations), n_sample, replace=False)
        locations = [locations[i] for i in idx]

    # Only parts that have sampled events — not all 10k parts
    pk_groups = defaultdict(list)
    for pk, eid in locations:
        pk_groups[pk].append(eid)

    features_list = []
    with h5py.File(mc_h5_path, "r") as mh, h5py.File(probs_h5_path, "r") as ph:
        for pk, eids in tqdm(pk_groups.items(), desc=f"Loading {ptype}"):
            data_raw  = mh[f"{ptype}/raw/data/{pk}/data"].astype(np.float32)
            ev_starts = mh[f"{ptype}/raw/ev_starts/{pk}/data"].astype(np.int64)
            probs     = ph[f"{ptype}/probs/{pk}/data"]
            for eid in eids:
                s, e = int(ev_starts[eid]), int(ev_starts[eid + 1])
                sig = probs[s:e] > threshold
                feats = data_raw[s:e][sig]
                if with_probs:
                    feats = np.column_stack([feats, probs[s:e][sig]])
                features_list.append(feats)

    print(f"  {ptype}: encoding {len(features_list)} events from {len(pk_groups)} parts")
    return _encode_features_list(features_list, norm_config, model, device,
                                  batch_size=batch_size, feats_with_probs=with_probs)


def _load_exp_enc_feats(
    exp_preds_h5, exp_h5_path,
    threshold, model, norm_config, with_probs, device,
    n_sample=10_000, batch_size=256, seed=42,
) -> np.ndarray:
    """Sample n_sample exp events, run sig-noise only on sampled parts, encode."""
    from collections import defaultdict
    from gplotnikov_sig_noise_models.k_nsol_labelneq0_da_hs128_k0p0001.sig_noise_model_v3 import (
        load_model as load_snmodel, predict_flat,
    )
    sn_model, _, sn_dev = load_snmodel(device="auto")

    rng = np.random.RandomState(seed)
    locations = []
    with h5py.File(exp_preds_h5, "r") as f:
        grp = f[EXP_KEY]
        parts = list(grp["event_ids"].keys())
        if 100<len(parts):
            parts_sel = rng.choice(len(parts), min(100, len(parts)), replace=False)
            parts = [parts[i] for i in parts_sel.tolist()]
        for pk in parts:
            eids = grp[f"event_ids/{pk}/data"][:]
            locations.extend([(pk, int(e)) for e in eids])
    print(len(locations))
    if len(locations) > n_sample:
        idx = rng.choice(len(locations), n_sample, replace=False)
        locations = [locations[i] for i in idx]

    # Only parts that have sampled events — exp has 28 parts so not a concern,
    # but kept consistent with MC version
    pk_groups = defaultdict(list)
    for pk, eid in locations:
        pk_groups[pk].append(eid)

    features_list = []
    with h5py.File(exp_h5_path, "r") as src:
        src_key = "exp_reco" if "exp_reco" in src else "exp"
        raw = src[src_key]["raw"]
        for pk, eids in tqdm(pk_groups.items(), desc="Loading exp"):
            data_raw  = raw[f"data/{pk}/data"][:].astype(np.float32)
            ev_starts = raw[f"ev_starts/{pk}/data"][:].astype(np.int64)
            probs = predict_flat(sn_model, data_raw, ev_starts,
                                  batch_size=batch_size, device=sn_dev, normalize=True)
            for eid in eids:
                s, e = int(ev_starts[eid]), int(ev_starts[eid + 1])
                sig = probs[s:e] > threshold
                feats = data_raw[s:e][sig]
                if with_probs:
                    feats = np.column_stack([feats, probs[s:e][sig]])
                features_list.append(feats)

    print(f"  exp: encoding {len(features_list)} events from {len(pk_groups)} parts")
    return _encode_features_list(features_list, norm_config, model, device,
                                  batch_size=batch_size, feats_with_probs=with_probs), locations


In [ ]:
N_UMAP = 10_000

print("Encoding muatm...")
enc_muatm = _load_mc_enc_feats(
    MC_PREDS_H5, MC_H5, PROBS_H5, "muatm_2020",
    THRESHOLD, model, norm_config, with_probs, dev, n_sample=N_UMAP
)
print("Encoding nuatm...")
enc_nuatm = _load_mc_enc_feats(
    MC_PREDS_H5, MC_H5, PROBS_H5, "nuatm_2020",
    THRESHOLD, model, norm_config, with_probs, dev, n_sample=N_UMAP
)
print("Encoding nue2...")
enc_nue2  = _load_mc_enc_feats(
    MC_PREDS_H5, MC_H5, PROBS_H5, "nue2_2020",
    THRESHOLD, model, norm_config, with_probs, dev, n_sample=N_UMAP
)
print("Encoding exp...")
enc_exp, locations  = _load_exp_enc_feats(
    EXP_PREDS_H5, EXP_H5,
    THRESHOLD, model, norm_config, with_probs, dev, n_sample=100_000
)
print(f"Encoder shapes: muatm={enc_muatm.shape}  nuatm={enc_nuatm.shape}  "
      f"nue2={enc_nue2.shape}  exp={enc_exp.shape}")

In [ ]:
strange_events = df_exp[EXP_MASK][['part_key', 'event_id']][df_exp[EXP_MASK]['score']>0.9]
strange_locations = strange_events.set_index(['part_key', 'event_id']).index.tolist()
strange_ids = []
for i, loc in enumerate(locations):
    if loc in strange_locations:
        strange_ids.append(i)

In [ ]:
import umap


enc_exp_strange = enc_exp[strange_ids]
all_enc = np.vstack([enc_muatm, enc_nuatm, enc_nue2, enc_exp, enc_exp_strange])
labels_umap = np.array(
    ["muatm_2020"] * len(enc_muatm) +
    ["nuatm_2020"] * len(enc_nuatm) +
    ["nue2_2020"]  * len(enc_nue2)  +
    ["exp_reco"]   * len(enc_exp) + 
    ["exp_reco_strange"]   * len(enc_exp_strange)
)

print(f"Running UMAP on {len(all_enc):,} events...")
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embedding = reducer.fit_transform(all_enc)
print("UMAP done.")

In [ ]:
plt.close()
with plt.rc_context(RC):
    fig, ax = plt.subplots(figsize=(8, 7))
    order = ["exp_reco", "muatm_2020", "nuatm_2020", "nue2_2020", "exp_reco_strange"]
    alpha = {"exp_reco": 0.8, "muatm_2020": 1.0, "nuatm_2020": 0.05, "nue2_2020": 0.05, "exp_reco_strange": 1.0}
    size  = {"exp_reco": 4,    "muatm_2020": 4,    "nuatm_2020": 6,    "nue2_2020": 6, "exp_reco_strange": 50}
    for lbl in order[:3]:
        mask = labels_umap == lbl
        ax.scatter(
            embedding[mask, 0], embedding[mask, 1],
            c=PTYPE_COLORS[lbl], label=PTYPE_LABELS[lbl],
            s=size[lbl], alpha=alpha[lbl], linewidths=0,
        )
    ax.set_title(f"UMAP of encoder representations")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.legend(markerscale=3)
    fig.tight_layout()
    plt.show()

## Cell 5 — MC metadata matching utility

Enrich MC predictions with truth info (energy, theta, …) from `baikal_mc_merged.h5`
by matching via `event_ids` stored in the predictions h5.

In [ ]:
def load_baikal_mc_merged_metadata(
    scores_h5_path: str | Path,
    mc_h5_path: str | Path,
    extra_keys: list | None = None,
) -> pd.DataFrame:
    """Load MC prediction scores and optionally match fields from mc_h5 via event_ids.

    Args:
        scores_h5_path: Output of predict_mc.py.
        mc_h5_path:     baikal_mc_merged.h5 (source of raw events).
        extra_keys:     List of (h5_dataset_template, col_name) pairs.
                        Template placeholders: {ptype}, {pk}.
                        Example:
                            [('{ptype}/prime_prty/{pk}/data', 'prime_prty')]
                        The fetched array is indexed by event_ids (rows only).

    Returns:
        DataFrame: ptype, part_key, event_id, score, n_sig_hits, n_sig_strings
                   + any extra_keys columns.
    """
    chunks = []
    with h5py.File(scores_h5_path, "r") as sh, h5py.File(mc_h5_path, "r") as mh:
        mc_grp = sh["baikal_mc_merged"]
        for ptype in mc_grp.keys():
            if "scores" not in mc_grp[ptype]:
                continue
            for pk in mc_grp[ptype]["scores"].keys():
                scores    = mc_grp[f"{ptype}/scores/{pk}/data"][:]
                n_sh      = mc_grp[f"{ptype}/n_sig_hits/{pk}/data"][:]
                n_ss      = mc_grp[f"{ptype}/n_sig_strings/{pk}/data"][:]
                event_ids = mc_grp[f"{ptype}/event_ids/{pk}/data"][:]
                df_part = pd.DataFrame({
                    "ptype":         ptype,
                    "part_key":      pk,
                    "event_id":      event_ids,
                    "score":         scores,
                    "n_sig_hits":    n_sh,
                    "n_sig_strings": n_ss,
                })
                if extra_keys:
                    for template, col_name in extra_keys:
                        ds_path = template.format(ptype=ptype, pk=pk)
                        if ds_path in mh:
                            df_part[col_name] = list(mh[ds_path][:][event_ids])
                chunks.append(df_part)
    df = pd.concat(chunks, ignore_index=True)
    print(f"MC metadata loaded: {len(df):,} events")
    return df


# Usage example:
# df_mc_meta = load_baikal_mc_merged_metadata(
#     MC_PREDS_H5, MC_H5,
#     extra_keys=[("{ptype}/prime_prty/{pk}/data", "prime_prty")],
# )
# df_mc_meta["energy_mc"] = df_mc_meta["prime_prty"].apply(lambda x: x[2])
# df_mc_meta["theta_mc"]  = df_mc_meta["prime_prty"].apply(lambda x: x[0])